# MoneyPrinterTurbo GitHub Issues queue — volledig geconfigureerd

Deze notebook:
- gebruikt nooit poort 8080;
- valideert ngrok en GitHub;
- vraagt verborgen om een LLM API-sleutel en een Pexels API-sleutel;
- schrijft deze alleen naar de tijdelijke `config.toml` in de Colab-runtime;
- start de MoneyPrinterTurbo API en GitHub Issues-worker;
- kan een mislukt Issue opnieuw in de wachtrij zetten.

Voer stap 1 t/m 4 in volgorde uit. Stap 5 is alleen voor diagnose.


## 1. Installeer de actuele fork

Deze cel zet een bestaande checkout altijd om naar jouw fork en haalt de actuele `main` op.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/Equilibriumpress/MoneyPrinterTurbo.git"
REPO_BRANCH = "main"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} bestaat, maar is geen Git-repository. "
        "Kies Runtime → Disconnect and delete runtime en probeer opnieuw."
    )
else:
    subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
            REPO_URL, str(REPO_DIR),
        ],
        check=True,
    )

worker_path = REPO_DIR / "scripts" / "github_issue_worker.py"
if not worker_path.is_file():
    raise RuntimeError(f"Queue-worker ontbreekt: {worker_path}")

os.chdir(REPO_DIR)
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "uv", "pyngrok", "toml"],
    check=True,
)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
subprocess.run(
    ["uv", "sync", "--frozen", "--python", "3.11"],
    check=True,
)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
origin = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
    text=True,
).strip()

print(f"Repository: {origin}")
print(f"Commit: {commit}")
print(f"Worker: {worker_path}")
print("Installatie gereed.")


## 2. Configureer ngrok, GitHub, AI en Pexels

Je hebt nodig:
- een fine-grained GitHub-token met `Issues: read and write`;
- een ngrok-token;
- één LLM API-sleutel, bijvoorbeeld Gemini, OpenAI of Moonshot;
- een Pexels API-sleutel omdat het test-Issue `video_source: pexels` gebruikt.

De invoer van sleutels blijft verborgen. De notebook schrijft ze uitsluitend naar de tijdelijke Colab-runtime.


In [ ]:
import json
import shutil
from getpass import getpass
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import toml
from pyngrok import ngrok

try:
    ngrok.kill()
except Exception:
    pass

ngrok_token = getpass("Enter the ngrok authentication token: " ).strip()
if not ngrok_token:
    raise ValueError("Een ngrok-token is verplicht")
ngrok.set_auth_token(ngrok_token)
del ngrok_token

github_token = getpass("Enter the fine-grained GitHub token: " ).strip()
if not github_token:
    raise ValueError("Een GitHub-token is verplicht")

def github_get(path):
    request = Request(
        f"https://api.github.com{path}",
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {github_token}",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "MoneyPrinterTurbo-Colab-Queue",
        },
    )
    try:
        with urlopen(request, timeout=30) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"GitHub-tokencontrole mislukt met HTTP {exc.code}: {detail}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"GitHub is niet bereikbaar: {exc}") from exc

profile = github_get("/user")
repository = github_get("/repos/Equilibriumpress/MoneyPrinterTurbo")
queued = github_get(
    "/repos/Equilibriumpress/MoneyPrinterTurbo/issues"
    "?state=open&labels=video-job&per_page=30"
)

providers = {
    "gemini", "openai", "moonshot",
    "deepseek", "groq", "pollinations",
}
provider = input(
    "LLM provider [gemini/openai/moonshot/deepseek/groq/pollinations] "
    "(standaard gemini): "
).strip().lower() or "gemini"
if provider not in providers:
    raise ValueError(f"Niet-ondersteunde provider: {provider}")

llm_api_key = getpass(f"Enter the {provider} API key: " ).strip()
if not llm_api_key:
    raise ValueError(f"Een API-sleutel voor {provider} is verplicht")

pexels_api_key = getpass("Enter the Pexels API key: " ).strip()
if not pexels_api_key:
    raise ValueError("Een Pexels API-sleutel is verplicht voor video_source=pexels")

model_override = input(
    "Optionele modelnaam (Enter gebruikt de standaard van MoneyPrinterTurbo): "
).strip()

config_path = REPO_DIR / "config.toml"
example_path = REPO_DIR / "config.example.toml"
if not config_path.exists():
    shutil.copyfile(example_path, config_path)

cfg = toml.load(config_path)
app_cfg = cfg.setdefault("app", {})
app_cfg["llm_provider"] = provider
app_cfg[f"{provider}_api_key"] = llm_api_key
app_cfg["pexels_api_keys"] = [pexels_api_key]
app_cfg["video_source"] = "pexels"

if model_override:
    app_cfg[f"{provider}_model_name"] = model_override
else:
    app_cfg[f"{provider}_model_name"] = ""

with config_path.open("w", encoding="utf-8") as handle:
    toml.dump(cfg, handle)

del llm_api_key
del pexels_api_key

queued_count = len([item for item in queued if "pull_request" not in item])

print(f"GitHub account: {profile.get('login')}")
print(f"Repository access: {repository.get('full_name')}")
print(f"Queued video jobs: {queued_count}")
print(f"LLM provider: {provider}")
print(f"Configuratie opgeslagen in: {config_path}")
print("De geheime sleutels zijn niet afgedrukt en bestaan alleen in deze Colab-runtime.")


## 3. Start de API en wachtrij

De cel kiest automatisch een vrije poort tussen 18080 en 18180 en raakt de Colab-systeempoort niet aan.


In [ ]:
import os
import socket
import subprocess
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

API_LOG_PATH = Path("/content/moneyprinterturbo-api-configured.log")
WORKER_LOG_PATH = Path("/content/moneyprinterturbo-github-worker-configured.log")

def stop_known_process(name):
    process = globals().get(name)
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)

stop_known_process("github_worker_proc")
stop_known_process("api_proc")

for log_name in ("github_worker_log", "api_log"):
    handle = globals().get(log_name)
    if handle is not None and not handle.closed:
        handle.close()

previous_api_tunnel = globals().get("api_tunnel")
if previous_api_tunnel is not None:
    try:
        ngrok.disconnect(previous_api_tunnel.public_url)
    except Exception:
        pass

def find_free_port(start=18080, end=18180):
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("127.0.0.1", port))
            except OSError:
                continue
            return port
    raise RuntimeError(f"Geen vrije poort gevonden tussen {start} en {end}")

API_PORT = find_free_port()
print(f"Veilige API-poort: {API_PORT}")

api_env = os.environ.copy()
api_env["PYTHONUNBUFFERED"] = "1"

api_log = API_LOG_PATH.open("w", encoding="utf-8")
api_proc = subprocess.Popen(
    [
        "uv", "run", "uvicorn", "app.asgi:app",
        "--host", "127.0.0.1",
        "--port", str(API_PORT),
        "--log-level", "warning",
    ],
    cwd=REPO_DIR,
    env=api_env,
    stdout=api_log,
    stderr=subprocess.STDOUT,
    text=True,
)

deadline = time.time() + 300
next_log_report = time.time() + 30
api_ready = False

while time.time() < deadline:
    try:
        with urlopen(
            f"http://127.0.0.1:{API_PORT}/openapi.json",
            timeout=2,
        ) as response:
            api_ready = response.status == 200
    except (URLError, TimeoutError):
        pass

    if api_ready or api_proc.poll() is not None:
        break

    if time.time() >= next_log_report:
        api_log.flush()
        recent = API_LOG_PATH.read_text(
            encoding="utf-8",
            errors="replace",
        )[-1600:]
        print("API start nog. Laatste log:")
        print(recent or "(nog geen loguitvoer)")
        next_log_report = time.time() + 30

    time.sleep(2)

if not api_ready:
    api_log.flush()
    recent_log = API_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )[-12000:]
    raise RuntimeError(
        "MoneyPrinterTurbo API kon niet starten.\n"
        f"Process return code: {api_proc.poll()}\n"
        f"Recente log:\n{recent_log}"
    )

api_tunnel = ngrok.connect(
    addr=f"http://127.0.0.1:{API_PORT}",
    proto="http",
    bind_tls=True,
)

worker_env = os.environ.copy()
worker_env["PYTHONUNBUFFERED"] = "1"
worker_env["MPT_GITHUB_TOKEN"] = github_token
worker_env["MPT_GITHUB_REPOSITORY"] = "Equilibriumpress/MoneyPrinterTurbo"
worker_env["MPT_API_BASE"] = f"http://127.0.0.1:{API_PORT}/api/v1"
worker_env["MPT_PUBLIC_BASE"] = api_tunnel.public_url
worker_env["MPT_MAX_VIDEO_COUNT"] = "1"

github_worker_log = WORKER_LOG_PATH.open("w", encoding="utf-8")
github_worker_proc = subprocess.Popen(
    [
        "uv", "run", "python", "-u",
        "scripts/github_issue_worker.py",
        "--poll-seconds", "20",
    ],
    cwd=REPO_DIR,
    env=worker_env,
    stdout=github_worker_log,
    stderr=subprocess.STDOUT,
    text=True,
)

worker_ready = False
deadline = time.time() + 45

while time.time() < deadline:
    if github_worker_proc.poll() is not None:
        break

    github_worker_log.flush()
    worker_text = WORKER_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )
    if "watches Equilibriumpress/MoneyPrinterTurbo" in worker_text:
        worker_ready = True
        break

    time.sleep(2)

if not worker_ready:
    github_worker_log.flush()
    recent_log = WORKER_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )[-12000:]
    raise RuntimeError(
        "GitHub-worker kon niet starten.\n"
        f"Process return code: {github_worker_proc.poll()}\n"
        f"Recente log:\n{recent_log}"
    )

print("GitHub Issues queue is running.")
print(f"Lokale API: http://127.0.0.1:{API_PORT}")
print(f"Tijdelijke resultaat-URL: {api_tunnel.public_url}")
print(f"API-documentatie: {api_tunnel.public_url}/docs")


## 4. Zet het mislukte Issue opnieuw in de wachtrij

Voer deze stap pas uit nadat stap 3 meldt dat de wachtrij draait. Standaard wordt Issue #2 opnieuw aangeboden.


In [ ]:
from urllib.request import Request, urlopen

issue_number_text = input("Issue-nummer om opnieuw te proberen (standaard 2): " ).strip() or "2"
issue_number = int(issue_number_text)

payload = json.dumps({"labels": ["video-job"]}).encode("utf-8")
request = Request(
    f"https://api.github.com/repos/Equilibriumpress/MoneyPrinterTurbo/issues/{issue_number}",
    data=payload,
    method="PATCH",
    headers={
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {github_token}",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "MoneyPrinterTurbo-Colab-Queue",
        "Content-Type": "application/json",
    },
)

try:
    with urlopen(request, timeout=30) as response:
        issue = json.loads(response.read().decode("utf-8"))
except HTTPError as exc:
    detail = exc.read().decode("utf-8", errors="replace")
    raise RuntimeError(
        f"Opnieuw in wachtrij zetten mislukt met HTTP {exc.code}: {detail}"
    ) from exc

print(f"Issue #{issue_number} staat opnieuw in de wachtrij.")
print(f"Labels: {[label['name'] for label in issue.get('labels', [])]}")
print("De worker controleert de wachtrij iedere 20 seconden.")


## 5. Diagnose

Voer deze cel uit wanneer een Issue niet naar `video-processing` verandert of wanneer de render stopt.


In [ ]:
def show_tail(path, title, length=12000):
    print(f"\n===== {title} =====\n")
    if not path.exists():
        print(f"Logbestand ontbreekt: {path}")
        return
    print(path.read_text(encoding="utf-8", errors="replace")[-length:])

print(f"API-proces actief: {globals().get('api_proc') is not None and api_proc.poll() is None}")
print(
    "Worker-proces actief: "
    f"{globals().get('github_worker_proc') is not None and github_worker_proc.poll() is None}"
)
print(f"API-poort: {globals().get('API_PORT', 'onbekend')}")
print(
    "Tunnel: "
    f"{getattr(globals().get('api_tunnel'), 'public_url', 'niet actief')}"
)

show_tail(API_LOG_PATH, "MoneyPrinterTurbo API")
show_tail(WORKER_LOG_PATH, "GitHub Issues worker")
